# 🌳 Архитектура ML-алгоритмов: Деревья решений и Ансамбли

Если линейные модели пытаются провести прямую линию через данные, то **алгоритмы на основе деревьев** рубят пространство на прямоугольные области. Они идеально находят сложные нелинейные закономерности (например, когда возраст до 30 лет влияет на покупку положительно, а после 30 — только если доход выше среднего).

---

## 1. Дерево решений (Decision Tree)

**Суть:** Алгоритм задает серии вопросов с ответами «Да/Нет» (например: `Возраст > 25?`, `Зарплата < 100k?`), чтобы максимально чисто разделить классы.

**Механизм (Как дерево выбирает, какой вопрос задать?):**
Дерево перебирает все возможные признаки и все возможные пороги (например, зарплату 50k, 51k, 52k...) и считает **Критерий информативности**. Чаще всего это **Примесь Джини (Gini Impurity)** или **Энтропия (Entropy)**.

Формула Энтропии (меры хаоса):
$$H = - \sum_{i=1}^{k} p_i \log_2(p_i)$$
Где $p_i$ — доля объектов $i$-го класса в текущей ветке.

*   Если в ветку попали 100 мошенников и 0 честных клиентов, энтропия $= 0$ (идеальный порядок, ветку можно больше не делить, это **лист**).
*   Если попали 50 мошенников и 50 честных, энтропия $= 1$ (максимальный хаос, нужно задавать следующий вопрос).

**Главная проблема:** Дерево невероятно жадное. Если его не ограничить (не задать максимальную глубину `max_depth`), оно вырастет до тех пор, пока каждый отдельный человек не окажется в своем собственном листе. Это называется **жестким переобучением (Overfitting)**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

# Создаем нелинейный датасет (в форме двух полумесяцев)
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)

# Обучаем два дерева: одно ограниченное, другое - переобученное
tree_good = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_moon, y_moon)
tree_overfitted = DecisionTreeClassifier(max_depth=None, random_state=42).fit(X_moon, y_moon)

print(f"Глубина правильного дерева: {tree_good.get_depth()} (находит общие правила)")
print(f"Глубина переобученного дерева: {tree_overfitted.get_depth()} (запоминает каждую точку наизусть)")

---

## 2. Случайный лес (Random Forest)

**Проблема:** Одно глубокое дерево слишком чувствительно к шуму. Если чуть-чуть изменить обучающие данные, оно построит совершенно другую структуру.

**Решение (Механика Bagging - Bootstrap Aggregating):**
Зачем использовать одного нестабильного гения, если можно собрать толпу из 1000 троечников, которые вместе примут идеальное решение?

1. **Случайные данные:** Из исходного датасета случайным образом (с повторениями) выбираются подвыборки. Каждое дерево видит только часть данных.
2. **Случайные признаки:** В каждом узле дерево выбирает вопрос не из всех 100 признаков, а только из случайной горстки (например, из $\sqrt{100} = 10$). Это делает деревья непохожими друг на друга (независимыми).
3. **Голосование:** Выращиваем 100 таких глубоких, переобученных деревьев. Для нового объекта каждое дерево выдает свой ответ (например, 70 проголосовали за «Класс 1», 30 за «Класс 0»). Лес принимает решение большинством голосов.

**Математический смысл:** Усреднение ответов множества независимых переобученных моделей резко снижает **дисперсию (Variance)** ошибки, не увеличивая смещение.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_moon, y_moon, test_size=0.3, random_state=42)

# Обучаем случайный лес из 100 деревьев
rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
rf.fit(X_train, y_train)

print(f"Точность одиночного дерева: {tree_overfitted.score(X_test, y_test):.2f}")
print(f"Точность Случайного леса: {rf.score(X_test, y_test):.2f}")
# Лес всегда стабильнее и точнее на новых данных!

---

## 3. Градиентный бустинг (Gradient Boosting / CatBoost / LightGBM)

**Суть:** Если Random Forest строит деревья *независимо и параллельно*, то Бустинг строит их **строго последовательно**. Каждое следующее дерево исправляет ошибки предыдущих.

**Механизм работы (Boosting):**
1. **Старт:** Создается базовая, самая примитивная модель (обычно константа — среднее значение или стартовая вероятность $0.5$).
2. **Оценка ошибок (Градиент):** Модель смотрит, где она ошиблась. Для каждого объекта вычисляется остаток: $Residual = y_{true} - y_{pred}$.
3. **Новое дерево:** Строится маленькое, неглубокое дерево (обычно глубина 3-6). Но предсказывает оно не сам таргет, а **именно эти остатки (ошибки)**!
4. **Шаг обучения (Learning Rate):** Предсказание нового дерева умножается на крошечный коэффициент $\eta$ (Learning Rate, например $0.05$) и прибавляется к общему ответу.
5. Процесс повторяется сотни раз.

**Математический смысл:** Бустинг делает градиентный спуск в пространстве функций (деревьев), а не весов. Новые деревья — это векторы, указывающие в сторону минимума функции потерь.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Обучаем градиентный бустинг
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

print(f"Точность Градиентного бустинга: {gb.score(X_test, y_test):.2f}")

---

## ⚖️ Итоговое сравнение алгоритмов Обучения с учителем

| Семейство алгоритмов | Главная логика | Плюсы | Минусы |
|---|---|---|---|
| **Линейные (Ridge/Lasso/LogReg)** | Ищут прямую зависимость, взвешивают признаки | Очень быстрые, понятные (видно веса), не склонны к переобучению | Плохо работают, если связи сложные и нелинейные |
| **Случайный лес (Random Forest)** | Голосование сотен независимых глубоких деревьев | Работает "из коробки", не требует настройки, не боится выбросов | Тяжелая модель (занимает много памяти), медленно предсказывает |
| **Градиентный бустинг (CatBoost)** | Эстафета мелких деревьев, исправляющих ошибки | Максимальная точность на таблицах, лидер Kaggle | Легко переобучить, требует тонкой настройки Learning Rate и глубины |